# QSP Project: Bactericidal vs Bacteriostatic Antibiotic Dynamics
## Core Model Demonstration

This notebook demonstrates the fundamental ODE model for bactericidal versus bacteriostatic antibiotics, including:
- Bacterial population dynamics (replicating, persister, SCV)
- Immune effector recruitment and killing
- Drug concentration-time profiles (PK)
- Cidal vs static mechanistic differences in sub-MIC exposure

In [ ]:
import sys
sys.path.insert(0, '../')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.gridspec import GridSpec

# Import our QSP modules
from qsp_static_cidal.parameters import get_default_parameters, get_drug_pk_parameters, normalize_pk_parameters
from qsp_static_cidal.pd_model import create_ode_system
from qsp_static_cidal.pk_models import TwoCompartmentPKModel, DosingRegimen
from qsp_static_cidal.simulation import run_simulation

print('Modules loaded successfully!')
print(f'NumPy version: {np.__version__}')
print(f'Matplotlib version: {plt.matplotlib.__version__}')

### Step 1: Load Default Parameters

In [ ]:
# Load default parameters
params = get_default_parameters()

print('Bacterial Parameters:')
for k, v in vars(params['bacteria']).items():
    print(f'  {k}: {v}')

print('
Immune Parameters:')
for k, v in vars(params['immune']).items():
    print(f'  {k}: {v}')

print('
Cytokine Parameters:')
for k, v in vars(params['cytokine']).items():
    print(f'  {k}: {v}')

### Step 2: Create PD Model

In [ ]:
# Create the bacterial + immune ODE system
pd_model = create_ode_system(params)

print('PD Model created:')
print(f'  Bacterial growth rate constant: {pd_model.p_bact.k_growth} per hour')
print(f'  Carrying capacity: {pd_model.p_bact.B_max:.1e} CFU/mL')
print(f'  Baseline immune effectors: {pd_model.p_imm.N_eff_0:.1e}')
print(f'  Alpha (cidal): {pd_model.p_cyto.alpha_cidal}, Alpha (static): {pd_model.p_cyto.alpha_static}')

### Step 3: Define Drug PK and Dosing Regimens

In [ ]:
# Define two archetypal drugs
weight = 70  # kg

# Doxycycline (bacteriostatic)
pk_static_raw = get_drug_pk_parameters('doxycycline')
pk_static = normalize_pk_parameters(pk_static_raw, weight)

print('Doxycycline (Static) PK Parameters (70 kg patient):')
for k, v in pk_static.items():
    print(f'  {k}: {v:.3f}')

# Meropenem (bactericidal)
pk_cidal_raw = get_drug_pk_parameters('meropenem')
pk_cidal = normalize_pk_parameters(pk_cidal_raw, weight)

print('
Meropenem (Cidal) PK Parameters (70 kg patient):')
for k, v in pk_cidal.items():
    print(f'  {k}: {v:.3f}')

### Step 4: Define Initial Conditions and Run Simulations

In [ ]:
# Initial condition: patient with respiratory infection
init_cond = {
    'B_rep': 1e6,        # 1 million CFU/mL replicating cells
    'B_pers': 1e3,       # small number of persisters
    'B_SCV': 0,          # no SCVs initially
    'N_eff': 1e7,        # baseline neutrophil capacity
    'Damage': 0,         # no cidal damage initially
    'IL6': 10,           # baseline IL-6
    'TNF': 5             # baseline TNF
}

print('Initial Conditions:')
for k, v in init_cond.items():
    if k in ['B_rep', 'B_pers', 'B_SCV']:
        print(f'  {k}: {v:.1e} CFU/mL')
    elif k == 'N_eff':
        print(f'  {k}: {v:.1e}')
    else:
        print(f'  {k}: {v:.1f}')

In [ ]:
# Define dosing regimens
# Doxycycline: 100 mg q12h (oral)
regimen_static = DosingRegimen(
    dose_mg=100,
    interval_hours=12,
    start_time=0,
    n_doses=8,
    infusion_duration_min=30  # minutes (for oral, represents absorption phase)
)

# Meropenem: 1000 mg q8h IV (30-min infusion)
regimen_cidal = DosingRegimen(
    dose_mg=1000,
    interval_hours=8,
    start_time=0,
    n_doses=12,
    infusion_duration_min=30
)

print('Dosing Regimens:')
print(f'  Doxycycline: {regimen_static.dose_mg} mg q{regimen_static.interval_hours}h (oral)')
print(f'  Meropenem: {regimen_cidal.dose_mg} mg q{regimen_cidal.interval_hours}h (IV)')
print(f'  Simulation duration: 96 hours')

In [ ]:
# Create PK models
pk_model_static = TwoCompartmentPKModel(**pk_static, effect_site_model=True)
pk_model_cidal = TwoCompartmentPKModel(**pk_cidal, effect_site_model=True)

print('PK models instantiated')
print(f'  Static (doxycycline) CL: {pk_static["CL"]} mL/min, Vc: {pk_static["Vc"]} mL')
print(f'  Cidal (meropenem) CL: {pk_cidal["CL"]} mL/min, Vc: {pk_cidal["Vc"]} mL')

In [ ]:
# Run simulations
print('
Running simulation: NO DRUG (control)...')
regimen_none = DosingRegimen(dose_mg=0, interval_hours=24, n_doses=0)
pk_model_none = TwoCompartmentPKModel(CL=1, Vc=1000, Vp=500, Q=100, Ka=0, Kp=1)
result_nodrug = run_simulation(pk_model_none, regimen_none, pd_model, init_cond, 
                               t_span=(0, 96), drug_class='none', weight_kg=weight)

print('Running simulation: DOXYCYCLINE (static)...')
result_static = run_simulation(pk_model_static, regimen_static, pd_model, init_cond, 
                               t_span=(0, 96), drug_class='static', weight_kg=weight)

print('Running simulation: MEROPENEM (cidal)...')
result_cidal = run_simulation(pk_model_cidal, regimen_cidal, pd_model, init_cond, 
                              t_span=(0, 96), drug_class='cidal', weight_kg=weight)

print('
Simulations complete!')

### Step 5: Visualization

In [ ]:
# Create comprehensive comparison figure
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

# --- Panel A: Bacterial burden ---
ax1 = fig.add_subplot(gs[0, :2])
t_no, B_no = result_nodrug.get_bacterial_burden()
t_sta, B_sta = result_static.get_bacterial_burden()
t_cid, B_cid = result_cidal.get_bacterial_burden()

ax1.semilogy(t_no, np.maximum(B_no, 1), 'k--', linewidth=2, label='No drug')
ax1.semilogy(t_sta, np.maximum(B_sta, 1), 'b-', linewidth=2, label='Doxycycline (static)')
ax1.semilogy(t_cid, np.maximum(B_cid, 1), 'r-', linewidth=2, label='Meropenem (cidal)')
ax1.set_xlabel('Time (hours)', fontsize=11)
ax1.set_ylabel('Total Bacterial Burden (CFU/mL)', fontsize=11)
ax1.set_title('A) Bacterial Population Dynamics', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)
ax1.set_ylim([1, 1e8])

# --- Panel B: Drug concentration at effect site ---
ax2 = fig.add_subplot(gs[0, 2])
C_eff_static = result_static.y[:, 3]
C_eff_cidal = result_cidal.y[:, 3]
ax2.plot(t_sta, C_eff_static, 'b-', linewidth=2, label='Doxycycline')
ax2.plot(t_cid, C_eff_cidal, 'r-', linewidth=2, label='Meropenem')
ax2.set_xlabel('Time (hours)', fontsize=11)
ax2.set_ylabel('Concentration (units)', fontsize=11)
ax2.set_title('B) Drug Conc. (Effect Site)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

# --- Panel C: Persister dynamics ---
ax3 = fig.add_subplot(gs[1, 0])
B_pers_no = result_nodrug.y[:, 1]
B_pers_sta = result_static.y[:, 1]
B_pers_cid = result_cidal.y[:, 1]
ax3.semilogy(t_no, np.maximum(B_pers_no, 1), 'k--', linewidth=2)
ax3.semilogy(t_sta, np.maximum(B_pers_sta, 1), 'b-', linewidth=2)
ax3.semilogy(t_cid, np.maximum(B_pers_cid, 1), 'r-', linewidth=2)
ax3.set_xlabel('Time (hours)', fontsize=11)
ax3.set_ylabel('Persisters (CFU/mL)', fontsize=11)
ax3.set_title('C) Persister Population', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# --- Panel D: SCV dynamics (resistance marker) ---
ax4 = fig.add_subplot(gs[1, 1])
B_scv_sta = result_static.y[:, 2]
B_scv_cid = result_cidal.y[:, 2]
ax4.plot(t_sta, B_scv_sta, 'b-', linewidth=2, label='Doxycycline')
ax4.plot(t_cid, B_scv_cid, 'r-', linewidth=2, label='Meropenem')
ax4.set_xlabel('Time (hours)', fontsize=11)
ax4.set_ylabel('SCV Population (CFU/mL)', fontsize=11)
ax4.set_title('D) SCV (Heteroresistance)', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend(fontsize=9)

# --- Panel E: Resistance fraction ---
ax5 = fig.add_subplot(gs[1, 2])
t_res_sta, frac_sta = result_static.get_resistance_fraction()
t_res_cid, frac_cid = result_cidal.get_resistance_fraction()
ax5.plot(t_res_sta, frac_sta, 'b-', linewidth=2, label='Doxycycline')
ax5.plot(t_res_cid, frac_cid, 'r-', linewidth=2, label='Meropenem')
ax5.set_xlabel('Time (hours)', fontsize=11)
ax5.set_ylabel('Fraction SCV/Total', fontsize=11)
ax5.set_title('E) Resistance Fraction', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.legend(fontsize=9)

# --- Panel F: Immune effectors ---
ax6 = fig.add_subplot(gs[2, 0])
N_no = result_nodrug.y[:, 7]
N_sta = result_static.y[:, 7]
N_cid = result_cidal.y[:, 7]
ax6.semilogy(t_no, N_no, 'k--', linewidth=2)
ax6.semilogy(t_sta, N_sta, 'b-', linewidth=2)
ax6.semilogy(t_cid, N_cid, 'r-', linewidth=2)
ax6.set_xlabel('Time (hours)', fontsize=11)
ax6.set_ylabel('Immune Effectors', fontsize=11)
ax6.set_title('F) Neutrophil Recruitment', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3)

# --- Panel G: IL-6 production ---
ax7 = fig.add_subplot(gs[2, 1])
IL6_no = result_nodrug.y[:, 9]
IL6_sta = result_static.y[:, 9]
IL6_cid = result_cidal.y[:, 9]
ax7.plot(t_no, IL6_no, 'k--', linewidth=2, label='No drug')
ax7.plot(t_sta, IL6_sta, 'b-', linewidth=2, label='Doxycycline')
ax7.plot(t_cid, IL6_cid, 'r-', linewidth=2, label='Meropenem')
ax7.set_xlabel('Time (hours)', fontsize=11)
ax7.set_ylabel('IL-6 (pg/mL)', fontsize=11)
ax7.set_title('G) IL-6 Cytokine Production', fontsize=12, fontweight='bold')
ax7.grid(True, alpha=0.3)
ax7.legend(fontsize=9)

# --- Panel H: Cidal damage ---
ax8 = fig.add_subplot(gs[2, 2])
Dmg_sta = result_static.y[:, 8]
Dmg_cid = result_cidal.y[:, 8]
ax8.plot(t_sta, Dmg_sta, 'b-', linewidth=2, label='Doxycycline')
ax8.plot(t_cid, Dmg_cid, 'r-', linewidth=2, label='Meropenem')
ax8.set_xlabel('Time (hours)', fontsize=11)
ax8.set_ylabel('Cidal Damage (units)', fontsize=11)
ax8.set_title('H) Accumulated Drug Damage', fontsize=12, fontweight='bold')
ax8.grid(True, alpha=0.3)
ax8.legend(fontsize=9)

fig.suptitle('QSP Model: Bactericidal vs Bacteriostatic Antibiotic Dynamics', 
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout()
plt.savefig('qsp_core_model_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure saved as qsp_core_model_demo.png')

### Step 6: Summary Statistics

In [ ]:
# Compute summary endpoints at 96 hours
summary = pd.DataFrame({
    'Scenario': ['No Drug', 'Doxycycline (Static)', 'Meropenem (Cidal)'],
    'Final Bacterial Burden (CFU/mL)': [
        B_no[-1],
        B_sta[-1],
        B_cid[-1]
    ],
    'Log10 Reduction': [
        np.log10(B_no[0] / max(B_no[-1], 1)),
        np.log10(B_sta[0] / max(B_sta[-1], 1)),
        np.log10(B_cid[0] / max(B_cid[-1], 1))
    ],
    'Peak IL-6 (pg/mL)': [
        IL6_no.max(),
        IL6_sta.max(),
        IL6_cid.max()
    ],
    'Final SCV Fraction': [
        0,
        B_scv_sta[-1] / (B_sta[-1] + 1e-6),
        B_scv_cid[-1] / (B_cid[-1] + 1e-6)
    ],
    'Peak Neutrophils': [
        N_no.max(),
        N_sta.max(),
        N_cid.max()
    ]
})

print('\n' + '='*100)
print('SUMMARY STATISTICS AT 96 HOURS')
print('='*100)
print(summary.to_string(index=False))
print('='*100)

### Interpretation

**Key Observations:**

1. **Bactericidal (Meropenem) Shows Rapid Crash:**
   - Initial near-normal growth followed by abrupt population collapse
   - Driven by accumulated cidal damage (panel H)
   - Rapid reduction in total CFU

2. **Bacteriostatic (Doxycycline) Shows Dose-Dependent Slowdown:**
   - Growth progressively retarded but not eliminated
   - Allows time for immune clearance
   - SCV emergence under prolonged static pressure

3. **Inflammatory Toxicity Trade-off:**
   - Cidal drug triggers higher peak IL-6 via TLR9-mediated DNA release
   - Static drug shows more modest cytokine elevation
   - Clinical implication: immunocompromised hosts may tolerate static better

4. **Resistance Evolution:**
   - SCV emergence most pronounced under doxycycline (static) selection
   - This matches experimental data showing SCV selection under prolonged static pressure

**This is the mechanistic basis for the static vs. cidal paradigm shift in your QSP proposal.**